# Méthode GéoVélo seul

Ce notebook montre la méthode GÉOVÉLO SEUL : la longueur du réseau cyclable d'une commune, prise
TELLE QUELLE dans les exports GéoVélo annuels - sans passer par OpenStreetMap ni par un modèle
(contrairement à FOB, voir `../02_FOB/`).

**Gardée pour comparaison** (voir `../README.md`) - N'est PAS utilisée par la suite du projet.

**Note sur cet environnement de démonstration** : aucun export GéoVélo brut n'est fourni avec ce dépôt
(source tierce, voir `00_transformation_des_donnees/README.md`). `longueur_geovelo.py` est donc montré
ici (section 1, code prêt à l'emploi), mais les VRAIS chiffres viennent de la table déjà vendorisée
(section 2).

## 0. Configuration

In [1]:
import sys
from pathlib import Path

ICI = Path.cwd()
sys.path.insert(0, str(ICI))
sys.path.insert(0, str(ICI.parent))
RACINE = ICI.parents[2]

import pandas as pd
pd.set_option("display.width", 160)

FICHIER_REFERENCE = RACINE / "data" / "donnees_brutes" / "reference" / "longueurs_reseau_reference.csv"
print("Référence :", FICHIER_REFERENCE)

Référence : C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_brutes\reference\longueurs_reseau_reference.csv


## 1. La fonction — [`longueur_geovelo.py`](longueur_geovelo.py)

`longueurs_par_commune()` calcule la longueur effective (Lambert-93, un côté bidirectionnel comptant
double) à partir d'UN export GéoVélo déjà chargé - voir sa docstring pour le détail. `preparer_longueurs_geovelo()`
l'applique à toutes les années disponibles dans un dossier d'exports.

In [2]:
from longueur_geovelo import lire_export, longueurs_par_commune, preparer_longueurs_geovelo

print(lire_export.__doc__)
print()
print(longueurs_par_commune.__doc__)

Un export GéoVélo annuel (parquet avec géométrie WKB, ou geojson) en tronçons WGS84.

Longueur effective par commune : 'code_commune, len_d, len_g' (km).

    'alpha' : multiplicateur d'un côté bidirectionnel (convention du projet : 2 - voir le docstring du
    module).


## 2. Le vrai résultat pour ce dépôt — [`reference.py`](../reference.py)

Sans export GéoVélo brut disponible, ce sont les longueurs déjà calculées (vendorisées) qu'on regarde
ici - colonnes `len_d_GV`/`len_g_GV` de la table de référence.

In [3]:
from reference import charger_reference, longueur_effective

reference = charger_reference(FICHIER_REFERENCE)
reference["km_effectif_GV"] = longueur_effective(reference, "GV")

par_annee = reference.groupby("annee")["km_effectif_GV"].sum().round(0)
print("Linéaire GéoVélo national, par année (km) :")
par_annee

Linéaire GéoVélo national, par année (km) :


annee
2019    120528.0
2020    120528.0
2021    124680.0
2022    106731.0
2023    114149.0
2024    126397.0
2025    134584.0
Name: km_effectif_GV, dtype: float64

## 3. Les communes au réseau GéoVélo le plus long (2024)

In [4]:
reference_2024 = reference[reference["annee"] == 2024]
reference_2024.nlargest(10, "km_effectif_GV")[["code_commune", "nom_commune", "code_departement", "km_effectif_GV"]]

,code_commune,nom_commune,code_departement,km_effectif_GV
68609,31555,Toulouse,31,1227.8004
74271,67482,Strasbourg,67,682.4145
70429,44109,Nantes,44,651.6174
68695,33063,Bordeaux,33,613.8691
69182,35238,Rennes,35,517.6458
72536,59350,Lille,59,489.7837
70888,49007,Angers,49,396.8096
69450,37261,Tours,37,386.1137
72407,59183,Dunkerque,59,379.4139
71227,51454,Reims,51,378.9549
